In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-3: RLAIF and Constitutional AI

## Where this session fits

> **The limitation we start from: RLHF needs people to compare completions, and people are slow and costly. Can a model do the comparing, given written principles?**

- **The question:** can a model, guided by written principles, supply the comparisons that people supplied in RLHF?
- **Position:** session 3 of 14
    - RLLM-2 built the reward from human comparisons
    - RLLM-4 opens the RL stage that uses it
- Reward sources: people compare (RLHF) (RLLM-2); **an AI compares (RLAIF) (RLLM-3)**; a program checks (RLVR) (RLLM-5) <!-- long-ok -->
- **Series goal:** turn a pretrained language model into a useful, aligned assistant with reinforcement learning, and know the methods well enough to choose among them

**Review from RLLM-2: the terms this session uses**

- a **preference example** is $(\x, \y^+, \y^-)$: a prompt, a preferred completion, a non-preferred completion
- a **reward model** is trained on preference examples and gives one reward per completion
- **RLHF** then runs RL against the reward model, with a KL penalty to the **reference model** (the frozen SFT model)
- **over-optimization:** the policy exploits the reward model's errors, and real quality falls

**By the end you can**

1. say what RLAIF changes in the RLHF pipeline, and what it keeps
2. read one critique and revision of an answer, and say why the model is not judging itself blindly
3. say how a model picks the better of two completions, and what a soft label is
4. name what each method gains and what it risks
5. name RLVR, the third reward source, and say what it needs


## RLAIF, the AI labeler and the constitution

> **RLAIF keeps the RLHF pipeline and replaces the human who compares completions by a language model.**

1. **RLAIF** is reinforcement learning from AI feedback
2. the **AI labeler** is the language model that does the comparing
3. a **constitution** is a short list of written principles for desired behavior
    - example: "choose the response that is less harmful, and does not encourage illegal or unethical activity" (paraphrased)
    - the actual list is longer, and it was used for harmlessness only ([Bai et al. 2022](https://arxiv.org/abs/2212.08073))
4. a **red-team prompt** is a prompt written to provoke a harmful completion

**Why the paper wanted a constitution** ([Bai et al. 2022](https://arxiv.org/abs/2212.08073))

- **scalable supervision:** as models grow more capable, use them to help supervise other models
- **fewer human labels:** the only human input for harmlessness is the list of principles
- **a written list:** the principles are explicit, so people can read and argue about them (our reading)
- [Lee et al. 2023](https://arxiv.org/abs/2309.00267) add that an AI labeler of the same size as the policy still beat the SFT baseline


## Road map: one constitution, two uses

> **The same written principles are used in two ways: Use 2 (pick between answers) is the core, and Use 1 (rewrite answers) is the extension that makes the starting policy.**

<img src="images/rllm3_roadmap.png" width=74%>

- **Use 1** is an extension: it makes SFT data, and the fine-tuned model is the starting policy for Use 2
- **Use 2** makes the preference data that RLHF needs, so Use 2 answers our question
- we show Use 1 first because the paper's pipeline runs in that order
- the paper's words are the supervised stage (Use 1) and the RL stage (Use 2)
    - SL-CAI and RL-CAI name the resulting models
- **Constitutional AI** is RLAIF with a labeler that is given written principles ([Bai et al. 2022](https://arxiv.org/abs/2212.08073))


## Use 1: rewrite answers, a worked example

**Use 1 is the extension that makes the starting policy: we fine-tune on answers the model rewrote itself. Here is one pass.**

<img src="images/rllm3_revise.png" width=58%>

- the loop is prompting: append the critique request, then the revision request, and let the model continue
- the SFT example is $(\x, \y')$: the prompt and the revised answer
    - the first answer $\y$ is discarded, and no preference pair is made
- the first model is already trained for helpfulness with human labels, so humans stay in the pipeline


## Use 1: why critique can work

**Why this slide?** One model just wrote, critiqued and revised. Why is that not circular?

> **The model is not asked to know more. It reads its own text against a principle now in the context.**

| role | input | output | the task |
|:--|:--|:--|:--|
| writer | the prompt | a whole answer | choose every word at once |
| critic | prompt, answer, and a written principle | the flaw, in words | compare a given text with one stated rule |

- judging a given answer is often easier than writing one at once (our reading)
    - Use 2's labeler relies on the same idea
- a blind spot of the model stays blind when it critiques itself (our reading)
- the paper reports that critique mattered mostly for smaller models


## Use 2: pick between answers

**Use 1 was the extension that makes the starting policy. The RL story runs through Use 2, which turns principles into preference labels.**

> **Use 2: the AI labeler picks the better of two answers, and a reward model learns from the picks.**

- a model writes two answers to each red-team prompt
    - in the paper they come from the model fine-tuned on Use 1's data, so they look like what RL will start from
- a labeler picks the better one by one principle, sampled for each comparison: that gives $(\x, \y^+, \y^-)$
    - the labeler is a separate language model that is not updated by RL (our reading of the setup)
- those pairs train a reward model, as in RLLM-2, and RL then proceeds as in RLHF, with the same risks

**How the labeler picks: a multiple-choice question**

<img src="images/rllm3_mcq.png" width=85%>

- a hard pick throws away how sure the labeler is
    - so the paper keeps a **soft label**: a probability such as 0.9, not a hard 1
- it is the labeler's log-probabilities of the tokens "A" and "B", **normalized** over just those two to sum to 1
- the figure is a paraphrase; the probabilities are invented

**From soft labels to a reward model**

**Why soft? A labeler 90% sure of its pick should teach the reward model more than one at 51%. A hard 0 or 1 cannot tell them apart.**

- let $\y^+$ be the completion the labeler favors, with probability $p$ (the soft label)
- let $d = \rew_\psi(\x, \y^+) - \rew_\psi(\x, \y^-)$ be the reward model's gap
- RLLM-2's loss took a hard winner
    - with $p$ it becomes $-p\log\sigma(d)-(1-p)\log(1-\sigma(d))$ (our arithmetic)
- a near-tie, $p$ close to 0.5, then pushes the reward model only weakly (our reading)


## A variant: score each completion, and skip the reward model

**Why this slide? The reward model is a second thing to train. Can the labeler supply the reward during RL?**

- [Lee et al. 2023](https://arxiv.org/abs/2309.00267) call it direct RLAIF: the labeler's score is the reward for each sampled completion
- it needs a score for one completion, not an A or B pick
    - the labeler rates the completion from 1 to 10, and the rating is averaged by the probability of each rating token
- the paper reports that it did at least as well as training a reward model first
- the cost: every RL sample needs a labeler call (our reading)


## What each method gains, and what it risks

**The choice between sources of preferences trades human time against how well the labels match what we want.**

| | RLHF | RLAIF | Constitutional AI |
|:--|:--|:--|:--|
| recipe | RL on human labels | RL on AI labels | RLAIF, plus an SFT stage (Use 1) |
| who compares | human annotators | a language model | a model, using a principle |
| written guidance | raters' guidelines | labeler instructions | a constitution |
| labels from AI | none | helpfulness, harmlessness, summaries | harmlessness only; helpfulness labels were human |
| cost | human time per label | model calls (no figure quoted here) | model calls, plus writing principles |

**What each method risks (our reading)**

- RLHF: raters may disagree
- RLAIF: the labeler's errors are copied, including position bias (it favors A or B, so swap and average) and self-preference (it favors its own style)
- Constitutional AI: a principle read too literally or too loosely, plus the RLAIF risks
- all three produce a reward model, so all three can be over-optimized (RLLM-2)
- evidence: [Lee et al. 2023](https://arxiv.org/abs/2309.00267) report RLAIF comparable to RLHF on summarization and dialogue tasks
    - they also report gains over SFT when the AI labeler is the same size as the policy
    - [Bai et al. 2022](https://arxiv.org/abs/2212.08073) report answers that are harmless without being evasive


## A third source: no labeler at all (RLVR)

**Why this slide? Humans and AI labelers both give opinions. Some tasks have an answer a program can check.**

> **RLVR: a program, a checker, supplies the reward. No person and no AI labeler compares anything.**

- **RLVR** is reinforcement learning with verifiable rewards
- it needs tasks with checkable answers, such as a math result or code that must pass tests
- the reward source (RLVR) is a separate choice from the optimizer (PPO or GRPO)
- a checker can be gamed too (RLLM-5)
- the reward-sources line on this deck's opening slide shows it: Reward sources: people compare (RLHF) (RLLM-2); **an AI compares (RLAIF) (RLLM-3)**; a program checks (RLVR) (RLLM-5) <!-- long-ok -->


# What we established, and what it sets up

- **Established**
    - RLAIF keeps the RLHF pipeline and replaces the human who compares completions by an AI labeler
    - Use 1 (rewrite answers) gives SFT data and the starting model
    - Use 2 (pick between answers) gives the preference data for a reward model
    - the labeler picks by a multiple-choice question about a principle, and its token probabilities become a soft label
    - a variant scores each completion and skips the reward model
    - a third source, RLVR, uses a program as the reward and needs checkable answers
    - AI labels move the risk from human cost to labeler errors, biases and how principles are read
- **This session's own limit:** we did not test whether AI labels agree with human ones
    - the papers do, and we only quote them
- **Sets up**
    - all three sources are now seen: human comparisons (RLLM-2), AI labels (this session), a checker (RLVR)
    - the RL stage consumes any of them, and a learned proxy can still be exploited
    - next, RLLM-4 (PPO in depth) opens that RL stage: what it runs on a language model, and where it costs or breaks
